<h2 align="center" style="color:blue">IMPORTING LIBRARY</h2>

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets,transforms,models
from torch.utils.data import DataLoader,random_split
from matplotlib import pyplot as plt
import optuna,time

In [2]:
device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


<h2 align="center" style="color:blue">LOAD DATASETS</h2>

In [3]:
transform=transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [4]:
dataset=datasets.ImageFolder(root='./dataset',transform=transform)
len(dataset)

2300

In [5]:
class_names=dataset.classes

In [6]:
num_clases=len(class_names)
num_clases

6

In [7]:
train_size=int(len(dataset)*0.75)
test_size=int(len(dataset) -train_size)
train_size,test_size

(1725, 575)

In [8]:
train_dataset,test_dataset=random_split(dataset,[train_size, test_size])

In [9]:
train_dataloader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_dataloader=DataLoader(test_dataset,batch_size=32,shuffle=True)

In [10]:
for images,labels in train_dataloader:
    print(images.shape)
    print(labels.shape)
    break

torch.Size([32, 3, 224, 224])
torch.Size([32])


In [11]:
labels[1]

tensor(3)

In [12]:
images[1].shape

torch.Size([3, 224, 224])

<h2 align="center" style="color:blue">TRANSFER LEARNING FROM RESNET</h2>

In [13]:
class CARClassifierResNet(nn.Module):
    def __init__(self,num_clases,dropout_rate=0.5):
        super().__init__()
        self.model=models.resnet50(weights='DEFAULT')
        for params in self.model.parameters():
            params.requires_grad=False

        for params in self.model.layer4.parameters():
            params.requires_grad=True 
               
        in_features = self.model.fc.in_features

        self.model.fc= nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(in_features,num_clases)
        )
    def forward(self,x):
        x=self.model(x)
        return x    

In [16]:
def train_one_epoch(model, optimizer, loss_fn):
    model.train()
    for images, labels in train_dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(images), labels)
        loss.backward()
        optimizer.step()

def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            predicted = model(images).argmax(1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

def objective(trial):
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.7)

    model = CARClassifierResNet(num_clases=num_clases, dropout_rate=dropout_rate).to(device)
    loss_fn = nn.CrossEntropyLoss().to(device)
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=1e-4
    )

    start = time.time()
    try:
        for epoch in range(5):                         # few epochs for tuning
            train_one_epoch(model, optimizer, loss_fn)
            accuracy = evaluate(model, test_dataloader)

            trial.report(accuracy, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        trial.set_user_attr("seconds", time.time() - start)
        return accuracy
    finally:
        del model, optimizer                           # free GPU memory
        torch.cuda.empty_cache()

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=1)
)
study.optimize(objective, n_trials=20)

print("Best validation accuracy:", study.best_value)
print("Best params:", study.best_params)

[I 2026-10-21 23:08:01,120] A new study created in memory with name: no-name-c67a196c-b6b8-4bc8-ba95-b4ec30a98d72
[I 2026-10-21 23:11:03,403] Trial 0 finished with value: 75.82608695652173 and parameters: {'lr': 0.0001329291894316216, 'dropout_rate': 0.675357153204958}. Best is trial 0 with value: 75.82608695652173.
[I 2026-10-21 23:14:06,825] Trial 1 finished with value: 76.8695652173913 and parameters: {'lr': 0.001570297088405539, 'dropout_rate': 0.49932924209851826}. Best is trial 1 with value: 76.8695652173913.
[I 2026-10-21 23:17:03,826] Trial 2 finished with value: 70.6086956521739 and parameters: {'lr': 2.9380279387035334e-05, 'dropout_rate': 0.27799726016810133}. Best is trial 1 with value: 76.8695652173913.
[I 2026-10-21 23:18:16,554] Trial 3 pruned. 
[I 2026-10-21 23:21:03,755] Trial 4 finished with value: 80.52173913043478 and parameters: {'lr': 0.0006358358856676254, 'dropout_rate': 0.5540362888980227}. Best is trial 4 with value: 80.52173913043478.
[I 2026-10-21 23:22:09,8

Best validation accuracy: 80.52173913043478
Best params: {'lr': 0.0006358358856676254, 'dropout_rate': 0.5540362888980227}
